# Day G1 — 바이오메디컬 AI 지도: 무엇을 배우고, 무엇을 만드는가

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/advanced/G-ai-biomed/day-G1.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/advanced/G-ai-biomed/day-G1.md)

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 예시로 이해하기

### 예시 1 — 같은 데이터, 판별 모델과 생성 모델

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(0)
# 가상 데이터: 두 군의 보폭 시간(초)과 보폭 변동계수(%)
n = 200
y = rng.integers(0, 2, n)                      # 0 = 대조군, 1 = 환자군(가상)
stride = rng.normal(1.05 + 0.08 * y, 0.06)
cv = rng.normal(2.0 + 1.2 * y, 0.6)
X = np.column_stack([stride, cv])
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)

disc = LogisticRegression().fit(X_tr, y_tr)   # 판별 모델: P(군 | 측정값)만 배운다
gen = GaussianNB().fit(X_tr, y_tr)            # 생성 모델: 군마다 P(측정값 | 군)을 배운다
print("판별(로지스틱) 정확도:", round(disc.score(X_te, y_te), 3))
print("생성(가우시안 NB) 정확도:", round(gen.score(X_te, y_te), 3))

# 생성 모델은 '새 가상 환자'를 뽑을 수 있다: 군 1의 평균·분산에서 샘플링
mu, var = gen.theta_[1], gen.var_[1]
fake = rng.normal(mu, np.sqrt(var), size=(5, 2))
print("군 1에서 뽑은 가상 측정값 5개:")
print(np.round(fake, 2))
print("군 1 실제 평균:", np.round(X_tr[y_tr == 1].mean(axis=0), 2))

### 예시 3 — 군집 점검: 실루엣, 안정성, 그리고 순환 논리

In [ ]:
import numpy as np
from scipy import stats
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(1)
# 가상 데이터: 세 '아형'이 숨어 있는 환자 90명 × 측정 4개
centers = np.array([[0, 0, 0, 0], [3, 3, 0, 0], [0, 3, 3, 0]])
X = np.vstack([rng.normal(c, 1.0, size=(30, 4)) for c in centers])
X = StandardScaler().fit_transform(X)

print("k  실루엣  부트스트랩 안정성(ARI 평균)")
for k in range(2, 6):
    labels = KMeans(n_clusters=k, n_init=10, random_state=0).fit_predict(X)
    sil = silhouette_score(X, labels)
    aris = []
    for b in range(20):                       # 환자를 다시 뽑아 군집을 반복
        idx = rng.choice(len(X), len(X), replace=True)
        lb = KMeans(n_clusters=k, n_init=10, random_state=b).fit(X[idx]).predict(X)
        aris.append(adjusted_rand_score(labels, lb))
    print(f"{k}  {sil:.3f}   {np.mean(aris):.3f}")

# 순환 논리의 함정: 아무 구조 없는 잡음을 둘로 나누고, 같은 변수로 두 군을 검정
noise = rng.normal(size=(100, 1))
lab = KMeans(n_clusters=2, n_init=10, random_state=0).fit_predict(noise)
t, p = stats.ttest_ind(noise[lab == 0, 0], noise[lab == 1, 0])
print(f"잡음을 군집한 뒤 t-검정 p = {p:.1e}  (구조가 없어도 '유의'하게 나온다)")